# Figure: Vapor composition during degassing

In [ ]:
from pathlib import Path
import numpy as np

results_directory = Path().resolve().parent / "Model_Outputs"
SAVE_FIG = True

## Import data and styling

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from helpers.plot_styles import (
    PLOTLY_TICK_LEN,
    PLOTLY_FONT,
    PLOTLY_TICK_FONTSIZE,
    SAMPLE_DISPLAY_NAMES,
    TOOL_COLORS_HEX,
    TOOL_LINE_STYLE,
)
from helpers.degassing_data import load_all_systems

# --- USER INPUTS --- #
SAMPLES = ["MORB", "Kilauea", "Fuego", "Fogo"]
TOOLS  = ["DCompress", "DCompress (IM)", "EVo", "MAGEC", "SulfurX", "VolFe", "VESIcal_Iacono"]

In [ ]:
systems = load_all_systems(SAMPLES, TOOLS, results_dir=results_directory)

## Build the figure

In [ ]:
# Row definitions: (DataFrame column, y-axis label).
Y_ROWS = [
    ("H2O_v_mf","log<sub>10</sub>[H<sub>2</sub>O<sup>vapor</sup>]"),
    ("CO2_v_mf", "log<sub>10</sub>[CO<sub>2</sub><sup>vapor</sup>]"),
    ("SO2_v_mf","log<sub>10</sub>[SO<sub>2</sub><sup>vapor</sup>]"),
    ("H2_v_mf","log<sub>10</sub>[H<sub>2</sub><sup>vapor</sup>]"),
    ("CO_v_mf", "log<sub>10</sub>[CO<sup>vapor</sup>]"),
    ("S2_v_mf","log<sub>10</sub>[S<sub>2</sub><sup>vapor</sup>]"),
    ("CH4_v_mf","log<sub>10</sub>[CH<sub>4</sub><sup>vapor</sup>]"),
    ("OCS_v_mf", "log<sub>10</sub>[OCS<sup>vapor</sup>]"),
    ("H2S_v_mf","log<sub>10</sub>[H<sub>2</sub>S<sup>vapor</sup>]"),
]

n_rows, n_cols = len(Y_ROWS), len(SAMPLES)
top_titles = [SAMPLE_DISPLAY_NAMES.get(s, s) for s in SAMPLES]
subplot_titles = top_titles + [""] * ((n_rows - 1) * n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    shared_xaxes=True, vertical_spacing=0.01, horizontal_spacing=0.05,
    subplot_titles=subplot_titles,
)

for r, (species,y_label) in enumerate(Y_ROWS, start=1):
    for c, sample in enumerate(SAMPLES, start=1):
        for tool in TOOLS:
            df = systems.get(sample, {}).get(tool)
            if df is None or "P_bars" not in df.columns:
                continue
            p_init = df["P_bars"].iloc[0]
            if p_init == 0:
                continue
            x_norm = df["P_bars"] / p_init
            fig.add_trace(
                go.Scatter(
                    mode="lines",
                    x=x_norm, y=np.log10(df[species]),
                    name=tool,
                    line=dict(color=TOOL_COLORS_HEX.get(tool, "#333"), width=2,
                              dash=TOOL_LINE_STYLE.get(tool, "solid"),
                              ),
                    showlegend=False,
                ),
                row=r, col=c,
            )
        fig.update_yaxes(title_text=y_label if c == 1 else None, row=r, col=c)
        if r == n_rows:
            fig.update_xaxes(title_text="P / P<sub>i<sub>", row=r, col=c, range=[0, None])

fig.update_layout(
    height=1300, width=1000,
    plot_bgcolor="white",
    margin=dict(t=40, r=30, l=60, b=50),
    font=PLOTLY_FONT,
)
fig.update_xaxes(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    ticks="outside", ticklen=PLOTLY_TICK_LEN, tickcolor="black",
    tickfont=dict(size=PLOTLY_TICK_FONTSIZE),
)
fig.update_yaxes(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    ticks="outside", ticklen=PLOTLY_TICK_LEN, tickcolor="black",
    tickfont=dict(size=PLOTLY_TICK_FONTSIZE),
    rangemode="tozero",
)

if SAVE_FIG:
    fig.write_image("figures/Fig_vapor_comp.png", scale=2, height=1300, width=1000)

fig.show()